# 🧠 CalRetail — Warehouse Slotting Optimisation
## Goal
Classify warehouse products by sales velocity and assign optimal travel storage zones.

## Algorithmic Explanation
**ABC Velocity Classification**
1. Measure item movements frequency using inventory_movements.
2. Sort warehouse inventories down cumulative demands.
3. Segment into Pareto categories (A: closest fast slots 20%, B: mid, C: far bulk storage).


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json
import re
import math
from backend.utils.db import load_table  # SQLite-backed
warnings.filterwarnings('ignore')

# Set path to include parent directory
base_path = Path().resolve()
while not (base_path / 'data').exists() and base_path.parent != base_path:
    base_path = base_path.parent
processed_dir = base_path / 'data'

print(f"Project root found at: {base_path}")
print(f"Data directory: {processed_dir}")


In [ ]:
movements = load_table('inventory_movements')
inv = load_table('inventory')

# Sum velocity per SKU by merging with inventory to resolve product_id
movements = pd.merge(movements, inv[['inventory_id', 'product_id']], on='inventory_id', how='left')
velocity = movements.groupby('product_id')['quantity'].sum().reset_index()
velocity = velocity.sort_values(by='quantity', ascending=False)

# Compute cumulative shares
velocity['cumulative_sales'] = velocity['quantity'].cumsum()
total_sales = velocity['quantity'].sum()
velocity['cum_pct'] = (velocity['cumulative_sales'] / total_sales) * 100

print(f"Aggregated SKU movements details. Top item sales share: {velocity['cum_pct'].iloc[0]:.2f}%")


In [ ]:
def compute_abc_slotting_plan(warehouse_id=None):
    global movements, inv
    
    # Filter inventory records for the specific warehouse if provided
    if warehouse_id:
        wh_inv = inv[inv['warehouse_id'] == warehouse_id]
        m = movements[movements['inventory_id'].isin(wh_inv['inventory_id'])]
    else:
        wh_inv = inv
        m = movements

    # Group movements by inventory_id to get total movement quantities
    velocity_mv = m.groupby('inventory_id')['quantity'].sum().reset_index()
    
    # Merge warehouse inventory with velocity movements
    wh_inv_vel = pd.merge(wh_inv[['inventory_id', 'product_id']], velocity_mv, on='inventory_id', how='left')
    wh_inv_vel['quantity'] = wh_inv_vel['quantity'].fillna(0)
    
    # Group by product_id (warehouse-specific velocity)
    prod_vel = wh_inv_vel.groupby('product_id')['quantity'].sum().reset_index()
    prod_vel = prod_vel.sort_values(by='quantity', ascending=False)
    
    # Calculate cumulative shares
    prod_vel['cumulative_sales'] = prod_vel['quantity'].cumsum()
    total_sales = prod_vel['quantity'].sum()
    if total_sales > 0:
        prod_vel['cum_pct'] = (prod_vel['cumulative_sales'] / total_sales) * 100
    else:
        prod_vel['cum_pct'] = 100.0

    results = []
    for idx, row in prod_vel.iterrows():
        pct = row['cum_pct']
        qty = row['quantity']
        
        if qty == 0:
            abc_class = 'C'
            zone = 'Zone 3 (Far Bulk Storage)'
        elif pct <= 80.0:
            abc_class = 'A'
            zone = 'Zone 1 (Golden Fast Pick)'
        elif pct <= 95.0:
            abc_class = 'B'
            zone = 'Zone 2 (Mid Distance)'
        else:
            abc_class = 'C'
            zone = 'Zone 3 (Far Bulk Storage)'
            
        results.append({
            "product_id": row['product_id'],
            "total_movements": int(qty),
            "cum_pct": round(float(pct), 2),
            "abc_class": abc_class,
            "assigned_zone": zone
        })
    return results

slotting_plan = compute_abc_slotting_plan()
print("Slotting output sample:\n", json.dumps(slotting_plan[0], indent=2))


In [ ]:
print("=== CALRETAIL WAREHOUSE CONTROLLER ===")
slot_df = pd.DataFrame(slotting_plan)
print("Aggregate Space Allocations count:")
print(slot_df['abc_class'].value_counts())
print("\nFast Movers (A Class) Assigned closest to dispatch:")
print(slot_df[slot_df['abc_class'] == 'A'][['product_id', 'total_movements', 'assigned_zone']].head(5).to_string(index=False))
